## Step 1: Import everything we need

In [2]:
import os
from dotenv import load_dotenv
## Here, we are importing our OS to import all the api keys and also giving the file paths
## dot env is the library imported from the module python-dotenv (installed in the requirements.txt or in the terminal)

from langchain_community.document_loaders import TextLoader #Loader
from langchain_text_splitters import RecursiveCharacterTextSplitter #Chunking
from langchain_community.embeddings import JinaEmbeddings #Embeddings
from langchain_community.vectorstores import FAISS #VectorDB

from langchain_groq import ChatGroq #LLM
from langchain.agents import create_agent #AI Agent



/var/folders/3q/fs45q76572q_n6vtxydtxylh0000gn/T/ipykernel_4705/4130524303.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader #Loader


In [3]:
load_dotenv()

True

##### Note: Now we need to load all the API keys to the notebook and assign them to variables so that we can use them if we need in the code

In [4]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

# printing the statement below as a confirmation of keys loaded properly

print("Environment variable loaded")

Environment variable loaded


#### Step 2: Loading the data

In [5]:
DATA_FILE_PATH = os.path.join("data", "AE_Contract.txt")

#### Step 3: Data Ingestion / Data Loading

Langchain - - - > Loaded in the first line

In [6]:
from langchain_community.document_loaders import TextLoader

## from langchain_community, we just imported out text loader

loader = TextLoader(DATA_FILE_PATH, encoding = "utf -8")
documents = loader.load()

## "utf-8" means lets write encoding so that the computer can understand...

## will print that loading is successful

print("DATA LOADED") # this is the confirmation
print("="*40) # this is to create the separation line
print(documents) # to check the source and metadata information

DATA LOADED
[Document(metadata={'source': 'data/AE_Contract.txt'}, page_content="\n======================================================================\nPAGE 1\n======================================================================\nC001 — Offshore Platform Engineering &\nFabrication Contract\nFictional training dataset — not a real commercial contract.\nContract ID\nC001\nContract Title\nOffshore Platform Engineering & Fabrication Contract\nContractor\nNorthSea Energy Solutions Ltd.\nClient\nAtlantic Offshore Resources Ltd.\nContract Value\nUSD 48,600,000\nEffective Date\n15 Jan 2026\nExpiry Date\n30 Sep 2028\nProject Location\nAberdeen, UK / North Sea Block A17\nDocument Version\n1.0\nContract Status\nActive\n\n======================================================================\nPAGE 2\n======================================================================\n1. Scope of Work\n• The Contractor shall provide engineering, procurement, fabrication, installation support, testing and d

In [7]:
## To check the length of the document like how many documents are present in the folder

len(documents)

1

##### We will now check the document information

In [8]:
print(documents[0].page_content)
print(documents[0].metadata)


PAGE 1
C001 — Offshore Platform Engineering &
Fabrication Contract
Fictional training dataset — not a real commercial contract.
Contract ID
C001
Contract Title
Offshore Platform Engineering & Fabrication Contract
Contractor
NorthSea Energy Solutions Ltd.
Client
Atlantic Offshore Resources Ltd.
Contract Value
USD 48,600,000
Effective Date
15 Jan 2026
Expiry Date
30 Sep 2028
Project Location
Aberdeen, UK / North Sea Block A17
Document Version
1.0
Contract Status
Active

PAGE 2
1. Scope of Work
• The Contractor shall provide engineering, procurement, fabrication, installation support, testing and documentation services
associated with the offshore platform engineering & fabrication contract.
• The work covers the assets and project area identified as Aberdeen, UK / North Sea Block A17. The Contractor shall maintain
a project execution plan, quality plan, inspection and test plan, and document register.
• Any work outside the defined scope requires a written Variation Order approved by au

In [9]:
## Lets get the total characters present in the text file
print(f"Total characters in document: {len(documents[0].page_content)}")

Total characters in document: 8406


#### Step 4: Data Splitting / Chunking

In [10]:
## Here we are importing the Recursive character text splitter from the langchain module (present in the requirements.txt file)
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 60
)

## Here, we have assigned the tool or weapons to cut the text like we did the setup for the splitting

chunks = text_splitter.split_documents(documents)

## Now, we got the chunks, present inside the variable "chunks"


In [12]:
## Now lets print one chunk and see if it worked
print(chunks)

[Document(metadata={'source': 'data/AE_Contract.txt'}, page_content='======================================================================\nPAGE 1\n======================================================================\nC001 — Offshore Platform Engineering &\nFabrication Contract\nFictional training dataset — not a real commercial contract.\nContract ID\nC001\nContract Title\nOffshore Platform Engineering & Fabrication Contract\nContractor\nNorthSea Energy Solutions Ltd.\nClient\nAtlantic Offshore Resources Ltd.\nContract Value\nUSD 48,600,000\nEffective Date\n15 Jan 2026\nExpiry Date\n30 Sep 2028\nProject Location\nAberdeen, UK / North Sea Block A17\nDocument Version\n1.0'), Document(metadata={'source': 'data/AE_Contract.txt'}, page_content='Aberdeen, UK / North Sea Block A17\nDocument Version\n1.0\nContract Status\nActive'), Document(metadata={'source': 'data/AE_Contract.txt'}, page_content='======================================================================\nPAGE 2\n============

In [13]:
# Lets see how many chunks we got
len(chunks)

18

In [14]:
# Now, lets see what is there inside one chunk
print(chunks[0])

page_content='======================================================================
PAGE 1
C001 — Offshore Platform Engineering &
Fabrication Contract
Fictional training dataset — not a real commercial contract.
Contract ID
C001
Contract Title
Offshore Platform Engineering & Fabrication Contract
Contractor
NorthSea Energy Solutions Ltd.
Client
Atlantic Offshore Resources Ltd.
Contract Value
USD 48,600,000
Effective Date
15 Jan 2026
Expiry Date
30 Sep 2028
Project Location
Aberdeen, UK / North Sea Block A17
Document Version
1.0' metadata={'source': 'data/AE_Contract.txt'}


In [15]:
# Lets see what is the page content in one chunk
print(chunks[0].page_content)

PAGE 1
C001 — Offshore Platform Engineering &
Fabrication Contract
Fictional training dataset — not a real commercial contract.
Contract ID
C001
Contract Title
Offshore Platform Engineering & Fabrication Contract
Contractor
NorthSea Energy Solutions Ltd.
Client
Atlantic Offshore Resources Ltd.
Contract Value
USD 48,600,000
Effective Date
15 Jan 2026
Expiry Date
30 Sep 2028
Project Location
Aberdeen, UK / North Sea Block A17
Document Version
1.0


#### Step 5: Embedding the chunks

In [16]:
## Now we will import libraries for embeddings
from langchain_community.embeddings import JinaEmbeddings

In [17]:
## Now we will set the embeddings model
embeddings_model = JinaEmbeddings(model_name = "jina-embeddings-v2-base-en")

## Now, lets print the confirmation
print("Embedding model ready and the name is ", embeddings_model.model_name)

Embedding model ready and the name is  jina-embeddings-v2-base-en


#### Step 6: Setting up, Vector Data Base

In [18]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings_model)
print("Chunks are stored", vector_store.index.ntotal)

Chunks are stored 18


#### Step 7: Store - Similarity Search

In [19]:
test_query = "What are the key commercial terms of the C001 contract?"

## Similaroty Search

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"Query: {test_query}\n")
for i, match in enumerate(top_matches, start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: What are the key commercial terms of the C001 contract?

--- Match 1 ---
15. Contract-Specific Commercial Schedule
Advance payment
10% against bank guarantee
Warranty
18 months from commissioning
Retention
5% until final acceptance
Currency
USD
End of fictional contract document.

--- Match 2 ---
PAGE 1
C001 — Offshore Platform Engineering &
Fabrication Contract
Fictional training dataset — not a real commercial contract.
Contract ID
C001
Contract Title
Offshore Platform Engineering & Fabrication Contract
Contractor
NorthSea Energy Solutions Ltd.
Client
Atlantic Offshore Resources Ltd.
Contract Value
USD 48,600,000
Effective Date
15 Jan 2026
Expiry Date
30 Sep 2028
Project Location
Aberdeen, UK / North Sea Block A17
Document Version
1.0



#### Step 8: Tool for the LLM Model

Note: This tool will be used later in the response code. This tool helps the AI agent to respond better and efficiently.
This is also called as the external super power of the AI agent.

In [20]:
retriever = vector_store.as_retriever(search_kwargs = {"k":3})
# to return 3 relevant documents

def aqe_chat_tool(question:str) -> str:
    """
    Search the contract document and give relevant answers...
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)


## we will get chunk.page_content for each chunks becz we will get 3 chunks


#### Step 9: LLM - Data retrieval pipeline

In [21]:
from langchain_groq import ChatGroq


llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

In [22]:
llm.model_name

# checking the model name is coming correct or not

'openai/gpt-oss-120b'

In [23]:
## Lets test the our LLM

test_response = llm.invoke("Hey, is learning RAG hard? Answer in 1 line")

In [24]:
print(test_response.content)

Learning RAG can be challenging at first, but with clear resources and hands‑on practice it becomes manageable.


#### Step 10: Creating our AI Agent

##### Note: AI Agent have 3 major things as below

1. Model - LLM
2. Tool - Super power (informally)
3. Memory - so here we are not using any memory at the moment



In [25]:
from langchain.agents import create_agent

aqe_assistant = create_agent(
    model = llm,
    tools = [aqe_chat_tool],
    system_prompt = """
    You are a Contract Intelligence Assistant. Your job is to answer questions about the provided oil and gas contract documents. 
    Use the aqe_chat_tool to retrieve relevant information from the contract documents before answering.
    Answer questions using only the information retrieved from the contract documents.
    Do not make up or assume information that is not present in the documents.
    If the information cannot be found in the documents, clearly say that the information was not found.
    Keep answers clear, concise, and professional.
    """
)

print("AQE assistant agent is ready to answer questions!")

AQE assistant agent is ready to answer questions!


In [26]:
def ask_aqe_assistant(question: str) -> str:
    """
    Send question to the RAG agent and print a nicely formatted answer.
    """

    print("=" * 60)
    print("Question:", question)
    print("-" * 60)

    response = aqe_assistant.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    answer = response["messages"][-1].content

    print("Answer:", answer)
    print("=" * 60)
    print()

    return answer

In [27]:
ask_aqe_assistant(
    "What are the key commercial terms of the C001 contract?"
)

Question: What are the key commercial terms of the C001 contract?
------------------------------------------------------------
Answer: **Key Commercial Terms – C001 Offshore Platform Engineering & Fabrication Contract**

| Item | Detail (as stated in the contract) |
|------|------------------------------------|
| **Contract Value** | USD 48,600,000 |
| **Currency** | United States Dollars (USD) |
| **Advance Payment** | 10 % of the contract value, payable against a bank guarantee |
| **Retention** | 5 % of each invoice, retained until final acceptance of the works |
| **Warranty** | 18 months from the date of commissioning |
| **Payment Milestones** | (Only the advance payment is explicitly defined in the commercial schedule; other milestones are not detailed in the excerpt provided.) |
| **Contract Duration** | Effective 15 Jan 2026 – Expiry 30 Sep 2028 |
| **Governing Parties** | Contractor – NorthSea Energy Solutions Ltd.; Client – Atlantic Offshore Resources Ltd. |

These terms con

'**Key Commercial Terms – C001 Offshore Platform Engineering & Fabrication Contract**\n\n| Item | Detail (as stated in the contract) |\n|------|------------------------------------|\n| **Contract Value** | USD\u202f48,600,000 |\n| **Currency** | United States Dollars (USD) |\n| **Advance Payment** | 10\u202f% of the contract value, payable against a bank guarantee |\n| **Retention** | 5\u202f% of each invoice, retained until final acceptance of the works |\n| **Warranty** | 18\u202fmonths from the date of commissioning |\n| **Payment Milestones** | (Only the advance payment is explicitly defined in the commercial schedule; other milestones are not detailed in the excerpt provided.) |\n| **Contract Duration** | Effective\u202f15\u202fJan\u202f2026 – Expiry\u202f30\u202fSep\u202f2028 |\n| **Governing Parties** | Contractor – NorthSea Energy Solutions Ltd.; Client – Atlantic Offshore Resources Ltd. |\n\nThese terms constitute the primary commercial provisions highlighted in the C001 contr